In [2]:
import os
import cv2

In [4]:

dataset_root = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_2/augmented_dataset_expansion_factor_0'

In [5]:
os.listdir(dataset_root)

['FD-030', 'FD-027', 'FD-029', 'FD-032', 'FD-031']

In [7]:
import os
import pandas as pd
import numpy as np
from PIL import Image

# ─── adjust this to the root of your unet_singan_augmented_datasets ───
dataset_root = '/home/miguel/GI/1.5 - Synthetic Data Generation/Singan-Seg/unet_singan_augmented_datasets_2/augmented_dataset_expansion_factor_0'

rows = []
for subject in os.listdir(dataset_root):
    subj_dir = os.path.join(dataset_root, subject)
    if not os.path.isdir(subj_dir):
        continue

    for split in ('train', 'val'):
        split_dir = os.path.join(subj_dir, split)
        if not os.path.isdir(split_dir):
            continue

        for fname in os.listdir(split_dir):
            if not fname.endswith('.png'):
                continue
            path = os.path.join(split_dir, fname)
            arr = np.array(Image.open(path))
            h, w = arr.shape[:2]
            px_min, px_max = int(arr.min()), int(arr.max())
            rows.append({
                'subject':   subject,
                'split':     split,
                'filename':  fname,
                'type':      'mask' if 'mask' in fname else 'image',
                'height':    h,
                'width':     w,
                'pixel_min': px_min,
                'pixel_max': px_max,
                'is_zero':   (px_max == 0)
            })

df = pd.DataFrame(rows)

# ─── per-file stats (optional) ───
print("Per-file stats:")
print(df.to_string(index=False))

# ─── per-subject & per-type summary ───
ps = (
    df
    .groupby(['subject', 'type'])
    .agg(
        total_count = ('filename', 'count'),
        zero_count  = ('is_zero',   'sum'),
    )
    .assign(
        zero_pct      = lambda d: d.zero_count  / d.total_count * 100,
        nonzero_count = lambda d: d.total_count - d.zero_count,
        nonzero_pct   = lambda d: 100 - d.zero_pct
    )
    .reset_index()
)
print("\nPer-subject & per-type zero stats:")
print(ps.to_string(index=False))

# ─── overall summary by type ───
ov = (
    df
    .groupby('type')
    .agg(
        total_count = ('filename', 'count'),
        zero_count  = ('is_zero',   'sum'),
    )
    .assign(
        zero_pct      = lambda d: d.zero_count  / d.total_count * 100,
        nonzero_count = lambda d: d.total_count - d.zero_count,
        nonzero_pct   = lambda d: 100 - d.zero_pct
    )
    .reset_index()
)
print("\nOverall zero stats by type:")
print(ov.to_string(index=False))

Per-file stats:
subject split                  filename  type  height  width  pixel_min  pixel_max  is_zero
 FD-030 train FD-032-slice-04-image.png image     256    256          0        255    False
 FD-030 train  FD-029-slice-19-mask.png  mask     256    256          0          0     True
 FD-030 train  FD-029-slice-48-mask.png  mask     256    256          0        255    False
 FD-030 train  FD-031-slice-43-mask.png  mask     256    256          0        255    False
 FD-030 train FD-027-slice-71-image.png image     256    256          0        255    False
 FD-030 train  FD-031-slice-16-mask.png  mask     256    256          0        255    False
 FD-030 train  FD-031-slice-26-mask.png  mask     256    256          0        255    False
 FD-030 train  FD-027-slice-52-mask.png  mask     256    256          0        255    False
 FD-030 train FD-027-slice-24-image.png image     256    256          0        255    False
 FD-030 train FD-027-slice-47-image.png image     256    256    

In [9]:
df['subject'].value_counts()

subject
FD-030    720
FD-027    720
FD-029    720
FD-032    720
FD-031    720
Name: count, dtype: int64

In [19]:
all_data = []
for split in ['train', 'val']:
    for subject in df['subject'].unique():
        subject_df = df[(df['subject'] == subject) & (df['split'] == split)]
        if subject_df.empty:
            print(f"No data for subject '{subject}' in split '{split}'")
            continue
        
        print(f"\nSubject: {subject}, Split: {split}")
        images = subject_df[subject_df['type'] == 'image']
        masks = subject_df[subject_df['type'] == 'mask']

        image_pixel_min = images['pixel_min'].min()
        image_pixel_max = images['pixel_max'].max()

        image_min_width = images['width'].min()
        image_max_width = images['width'].max()
        image_min_height = images['height'].min()
        image_max_height = images['height'].max()

        mask_pixel_min = masks['pixel_min'].min()
        mask_pixel_max = masks['pixel_max'].max()
        
        mask_min_width = masks['width'].min()
        mask_max_width = masks['width'].max()
        mask_min_height = masks['height'].min()
        mask_max_height = masks['height'].max()

        percent_images_zero = (images['is_zero'].sum() / len(images))
        percent_masks_zero = (masks['is_zero'].sum() / len(masks))

        all_data.append({
            'subject': subject,
            'split': split,
            'image_pixel_min': image_pixel_min,
            'image_pixel_max': image_pixel_max,

            'image_min_width': image_min_width,
            'image_max_width': image_max_width,
            'image_min_height': image_min_height,
            'image_max_height': image_max_height,

            'mask_pixel_min': mask_pixel_min,
            'mask_pixel_max': mask_pixel_max,

            'mask_min_width': mask_min_width,
            'mask_max_width': mask_max_width,
            'mask_min_height': mask_min_height,
            'mask_max_height': mask_max_height,

            'percent_images_zero': percent_images_zero,
            'percent_masks_zero': percent_masks_zero,
        })


Subject: FD-030, Split: train

Subject: FD-027, Split: train

Subject: FD-029, Split: train

Subject: FD-032, Split: train

Subject: FD-031, Split: train

Subject: FD-030, Split: val

Subject: FD-027, Split: val

Subject: FD-029, Split: val

Subject: FD-032, Split: val

Subject: FD-031, Split: val


In [20]:
pd.DataFrame(all_data)

,subject,split,image_pixel_min,image_pixel_max,image_min_width,image_max_width,image_min_height,image_max_height,mask_pixel_min,mask_pixel_max,mask_min_width,mask_max_width,mask_min_height,mask_max_height,percent_images_zero,percent_masks_zero
0,FD-030,train,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.343750
1,FD-027,train,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.364583
2,FD-029,train,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.357639
3,FD-032,train,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.364583
4,FD-031,train,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.375000
5,FD-030,val,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.430556
6,FD-027,val,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.347222
7,FD-029,val,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.375000
8,FD-032,val,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.347222
9,FD-031,val,0,255,256,256,256,256,0,255,256,256,256,256,0.0,0.305556


,subject,split,filename,type,height,width,pixel_min,pixel_max,is_zero
3459,FD-031,val,FD-031-slice-16-image.png,image,256,256,0,255,False
3462,FD-031,val,FD-031-slice-27-image.png,image,256,256,0,255,False
3464,FD-031,val,FD-031-slice-07-image.png,image,256,256,0,255,False
3467,FD-031,val,FD-031-slice-65-image.png,image,256,256,0,255,False
3469,FD-031,val,FD-031-slice-53-image.png,image,256,256,0,255,False
...,...,...,...,...,...,...,...,...,...
3590,FD-031,val,FD-031-slice-32-image.png,image,256,256,0,255,False
3592,FD-031,val,FD-031-slice-19-image.png,image,256,256,0,255,False
3594,FD-031,val,FD-031-slice-42-image.png,image,256,256,0,255,False
3597,FD-031,val,FD-031-slice-33-image.png,image,256,256,0,255,False


In [17]:
ps

,subject,type,total_count,zero_count,zero_pct,nonzero_count,nonzero_pct
0,FD-027,image,360,0,0.000000,360,100.000000
1,FD-027,mask,360,130,36.111111,230,63.888889
2,FD-029,image,360,0,0.000000,360,100.000000
3,FD-029,mask,360,130,36.111111,230,63.888889
4,FD-030,image,360,0,0.000000,360,100.000000
5,FD-030,mask,360,130,36.111111,230,63.888889
6,FD-031,image,360,0,0.000000,360,100.000000
7,FD-031,mask,360,130,36.111111,230,63.888889
8,FD-032,image,360,0,0.000000,360,100.000000
9,FD-032,mask,360,130,36.111111,230,63.888889


In [18]:
df

,subject,split,filename,type,height,width,pixel_min,pixel_max,is_zero
0,FD-030,train,FD-032-slice-04-image.png,image,256,256,0,255,False
1,FD-030,train,FD-029-slice-19-mask.png,mask,256,256,0,0,True
2,FD-030,train,FD-029-slice-48-mask.png,mask,256,256,0,255,False
3,FD-030,train,FD-031-slice-43-mask.png,mask,256,256,0,255,False
4,FD-030,train,FD-027-slice-71-image.png,image,256,256,0,255,False
...,...,...,...,...,...,...,...,...,...
3595,FD-031,val,FD-031-slice-23-mask.png,mask,256,256,0,255,False
3596,FD-031,val,FD-031-slice-52-mask.png,mask,256,256,0,255,False
3597,FD-031,val,FD-031-slice-33-image.png,image,256,256,0,255,False
3598,FD-031,val,FD-031-slice-26-image.png,image,256,256,0,255,False
